In [15]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 68.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 115.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [3]:
from datasets import load_dataset

ds = load_dataset("alzoubi36/opp_115")
ds


/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)
Generating test split: 100%|██████████| 697/697 [00:00<00:00, 119450.43 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2185
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 550
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 697
    })
})

In [4]:
ds["train"][0]

{'text': ' ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to your questions and comments. You may choose to provide additional information as well. ',
 'label': [3]}

In [5]:
len(ds["train"]), len(ds["validation"]), len(ds["test"])

(2185, 550, 697)

In [6]:
import requests, re, json
from pathlib import Path

Path("data").mkdir(exist_ok=True)
OUT_HTML = "data/gdpr_raw.html"
OUT_CLAUSES = "data/gdpr_clauses.jsonl"

# 1) Download GDPR page (simple HTML view)
url = "https://gdpr-info.eu/"  # human-readable page listing articles
print("Downloading:", url)
r = requests.get(url, timeout=30)
r.raise_for_status()
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(r.text)
print("Saved raw HTML to", OUT_HTML)

# 2) Simple split by "Article <number>" occurrences
text = r.text

# Replace some HTML tags with spaces to reduce garbage
clean = re.sub(r"<(script|style)[^>]*>.*?</\\1>", " ", text, flags=re.S|re.I)
clean = re.sub(r"<[^>]+>", " ", clean)     # strip remaining tags
clean = re.sub(r"\s+", " ", clean).strip() # collapse whitespace

parts = re.split(r"(Article\s+\d+)", clean, flags=re.I)
clauses = []
for i in range(1, len(parts), 2):
    title = parts[i].strip()
    body = parts[i+1].strip() if (i+1) < len(parts) else ""
    # keep only reasonably sized bodies
    if len(body) > 20:
        clauses.append({"id": title, "text": body[:400] + ("..." if len(body)>400 else "")})

# 3) Save clauses
with open(OUT_CLAUSES, "w", encoding="utf-8") as f:
    for c in clauses:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"Wrote {len(clauses)} clauses to {OUT_CLAUSES}")

# 4) Print first 3 for inspection
print("\nFirst 3 clauses (id => snippet):\n")
for i, c in enumerate(clauses[:3], start=1):
    print(f"{i}) {c['id']} => {c['text'][:300]}")


Downloading: https://gdpr-info.eu/
Saved raw HTML to data/gdpr_raw.html
Wrote 77 clauses to data/gdpr_clauses.jsonl

First 3 clauses (id => snippet):

1) Article 1 => Subject-matter and objectives
2) Article 4 => Definitions Chapter 2 Principles
3) Article 5 => Principles relating to processing of personal data


In [10]:
# Inspect one GDPR clause and three OPP-115 examples
import json
from pathlib import Path
from datasets import load_dataset

# load OPP-115 (we already loaded earlier into variable 'ds' possibly, but reload safely)
try:
    ds = load_dataset("alzoubi36/opp_115")
except Exception:
    # fallback: read the file we saved earlier if present
    ds = None

# load GDPR clauses file we created
gdpr_path = Path("data/gdpr_clauses.jsonl")
gdpr_clauses = []
if gdpr_path.exists():
    with open(gdpr_path, "r", encoding="utf-8") as f:
        for line in f:
            gdpr_clauses.append(json.loads(line))
else:
    print("gdpr_clauses.jsonl not found in data/ — please run the download cell again.")

print("=== GDPR clauses found:", len(gdpr_clauses), "===\n")

# Print the first GDPR clause (full text)
if len(gdpr_clauses) > 0:
    print("First GDPR clause (id):", gdpr_clauses[0].get("id"))
    print("Full text:\n", gdpr_clauses[0].get("text"))
else:
    print("No GDPR clauses to show.")

print("\n\n=== Showing 3 OPP-115 train examples ===\n")
if ds is not None:
    split = ds["train"]
    for i in range(3):
        ex = split[i]
        text = ex.get("text") or ex.get("segment") or ex.get("policy") or "<no text field>"
        label = ex.get("label")
        print(f"Example {i+1} — label: {label}\n{text}\n{'-'*60}\n")
else:
    # try to read saved sample file if present
    sample_path = Path("data/opp115_sample.jsonl")
    if sample_path.exists():
        with open(sample_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 3:
                    break
                rec = json.loads(line)
                print(f"Example {i+1} — meta: {rec.get('meta')}\n{rec.get('text')}\n{'-'*60}\n")
    else:
        print("OPP-115 not available; run the earlier loading cell first.")


=== GDPR clauses found: 77 ===

First GDPR clause (id): Article 1
Full text:
 Subject-matter and objectives


=== Showing 3 OPP-115 train examples ===

Example 1 — label: [3]
 ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to your questions and comments. You may choose to provide additional information as well. 
------------------------------------------------------------

Example 2 — label: [9]
 (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their use in their discretion, including direct marketing. Some of our contests and sweepstakes will ask you at the time of entry whether you would like to have your personal information shared with the sponsor, in which case we will honor your selection. Othe

In [16]:
# Simple TF-IDF retrieval demo: find top-3 GDPR clauses for a chosen OPP example
import json
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from datasets import load_dataset

# Load GDPR clauses we created
gdpr_path = Path("data/gdpr_clauses.jsonl")
gdpr_clauses = []
with open(gdpr_path, "r", encoding="utf-8") as f:
    for line in f:
        gdpr_clauses.append(json.loads(line))

gdpr_texts = [c["text"] for c in gdpr_clauses]
gdpr_ids = [c["id"] for c in gdpr_clauses]

# Load OPP-115 (already available)
ds = load_dataset("alzoubi36/opp_115")
example = ds["train"][1]   # uses Example 2 you printed (index 1)
query_text = example.get("text") or example.get("segment") or example.get("policy")

print("Query (truncated):\n", query_text[:400], "\n\n---\n")

# Build TF-IDF on GDPR clauses (simple)
vect = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=5000)
X = vect.fit_transform(gdpr_texts)

# Vectorize query and compute cosine similarities
qv = vect.transform([query_text])
cosine_similarities = linear_kernel(qv, X).flatten()

# Get top 3 clause indices
import numpy as np
top_idx = np.argsort(-cosine_similarities)[:3]

print("Top 3 GDPR clause matches:")
for rank, idx in enumerate(top_idx, start=1):
    print(f"\nRank {rank}: Clause id = {gdpr_ids[idx]}")
    print(f"Score = {cosine_similarities[idx]:.4f}")
    snippet = gdpr_texts[idx][:400].replace("\n"," ")
    print("Snippet:", snippet)


Query (truncated):
  (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their use in their discretion, including direct marketing. Some of our contests and sweepstakes will ask you at the time of entry whether you would like to have your personal information shared with the sp 

---

Top 3 GDPR clause matches:

Rank 1: Clause id = Article 12
Score = 0.3183
Snippet: Transparent information, communication and modalities for the exercise of the rights of the data subject Section 2 Information and access to personal data

Rank 2: Clause id = Article 85
Score = 0.2682
Snippet: Processing and freedom of expression and information

Rank 3: Clause id = Article 14
Score = 0.2593
Snippet: Information to be provided where personal data have not been obtained from the data subject


In [17]:
# Week 2 - Step 1: Run simple retrieval for 5 policy examples

import json
import numpy as np
from pathlib import Path
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Load GDPR clauses
gdpr_clauses = []
with open("data/gdpr_clauses.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        gdpr_clauses.append(json.loads(line))

gdpr_texts = [c["text"] for c in gdpr_clauses]
gdpr_ids = [c["id"] for c in gdpr_clauses]

# Build TF-IDF index once
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=5000)
X = vectorizer.fit_transform(gdpr_texts)

# Load OPP-115
ds = load_dataset("alzoubi36/opp_115")

print("Running retrieval for 5 OPP-115 examples...\n")

for i in range(5):
    ex = ds["train"][i]
    query_text = ex["text"]

    # Vectorize query
    qv = vectorizer.transform([query_text])
    scores = linear_kernel(qv, X).flatten()

    # Top 2 GDPR clauses
    top_idx = np.argsort(-scores)[:2]

    print(f"Example {i+1}")
    print("Policy text:", query_text[:200], "...")
    print("Retrieved GDPR clauses:")
    for idx in top_idx:
        print(f"  - {gdpr_ids[idx]} (score={scores[idx]:.3f})")
    print("-" * 60)


Running retrieval for 5 OPP-115 examples...

Example 1
Policy text:  ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to you ...
Retrieved GDPR clauses:
  - Article 12 (score=0.332)
  - Article 85 (score=0.303)
------------------------------------------------------------
Example 2
Policy text:  (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their ...
Retrieved GDPR clauses:
  - Article 12 (score=0.318)
  - Article 85 (score=0.268)
------------------------------------------------------------
Example 3
Policy text:  *Web Beacons: Military Web pages and the Web pages of our partners also utilize electronic images known as Web beacons (sometimes called single-pixel gifs, clear gifs or action tags) that a

In [18]:
# Small diagnostic: retrieve top 5 and show all
import json
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Load GDPR clauses
gdpr_clauses = []
with open("data/gdpr_clauses.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        gdpr_clauses.append(json.loads(line))

gdpr_texts = [c["text"] for c in gdpr_clauses]
gdpr_ids = [c["id"] for c in gdpr_clauses]

# TF-IDF
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=5000)
X = vectorizer.fit_transform(gdpr_texts)

ds = load_dataset("alzoubi36/opp_115")

ex = ds["train"][1]  # the third-party sharing example
query_text = ex["text"]

qv = vectorizer.transform([query_text])
scores = linear_kernel(qv, X).flatten()
top_idx = np.argsort(-scores)[:5]

print("Policy text (short):")
print(query_text[:200], "\n")

print("Top 5 GDPR clauses:")
for idx in top_idx:
    print(f"- {gdpr_ids[idx]} (score={scores[idx]:.3f})")


Policy text (short):
 (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their 

Top 5 GDPR clauses:
- Article 12 (score=0.318)
- Article 85 (score=0.268)
- Article 14 (score=0.259)
- Article 13 (score=0.259)
- Article 67 (score=0.216)


In [1]:
# Fix + fallback: upgrade sentence-transformers and use a reliable model
import sys
!{sys.executable} -m pip install -q --upgrade "sentence-transformers" "transformers>=4.30.0" || true

from sentence_transformers import SentenceTransformer, util
import json, numpy as np
from datasets import load_dataset
from pathlib import Path

# Load GDPR clauses
gdpr_clauses = []
with open("data/gdpr_clauses.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        gdpr_clauses.append(json.loads(line))
gdpr_texts = [c["text"] for c in gdpr_clauses]
gdpr_ids = [c["id"] for c in gdpr_clauses]

# Load an OPP example (same as before)
ds = load_dataset("alzoubi36/opp_115")
ex = ds["train"][1]
query_text = ex["text"]

print("Query (truncated):\n", query_text[:300], "\n\n---\n")

# Try to load a robust small model
MODEL_CANDIDATES = [
    "sentence-transformers/all-mpnet-base-v2",  # common name with prefix
    "all-mpnet-base-v2",
    "sentence-transformers/all-MiniLM-L6-v2",  # fallback, light & fast
    "all-MiniLM-L6-v2"
]

model = None
for name in MODEL_CANDIDATES:
    try:
        print("Trying to load model:", name)
        model = SentenceTransformer(name)
        print("Loaded:", name)
        break
    except Exception as e:
        print("Failed to load", name, "->", str(e).splitlines()[0])

if model is None:
    raise RuntimeError("Could not load any fallback sentence-transformer models. Check network or package versions.")

# Compute embeddings and dense similarities
emb_g = model.encode(gdpr_texts, convert_to_numpy=True, show_progress_bar=True)
emb_q = model.encode([query_text], convert_to_numpy=True)

# Normalize and compute cosine sims
from numpy.linalg import norm
emb_g = emb_g / np.maximum(1e-12, np.linalg.norm(emb_g, axis=1, keepdims=True))
emb_q = emb_q / np.maximum(1e-12, np.linalg.norm(emb_q, axis=1, keepdims=True))
sims = (emb_q @ emb_g.T)[0]

top5 = np.argsort(-sims)[:5]
print("\nTop 5 GDPR clauses by dense similarity:")
for r, idx in enumerate(top5, start=1):
    print(f"{r}) {gdpr_ids[idx]} (score={sims[idx]:.4f})")
    print("   ", gdpr_texts[idx][:300].strip(), "\n")


/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


Query (truncated):
  (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their use in their discretion, including direct marketing. Some of our contests and sweepstakes will ask  

---

Trying to load model: sentence-transformers/all-mpnet-base-v2
Loaded: sentence-transformers/all-mpnet-base-v2


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.15s/it]



Top 5 GDPR clauses by dense similarity:
1) Article 13 (score=0.4695)
    Information to be provided where personal data are collected from the data subject 

2) Article 14 (score=0.4694)
    Information to be provided where personal data have not been obtained from the data subject 

3) Article 90 (score=0.4024)
    Obligations of secrecy 

4) Article 67 (score=0.3933)
    Exchange of information Section 3 European data protection board 

5) Article 12 (score=0.3634)
    Transparent information, communication and modalities for the exercise of the rights of the data subject Section 2 Information and access to personal data 



In [2]:
# Week 2: BM25 retrieval for multiple OPP-115 examples (one cell)
import sys
!{sys.executable} -m pip install -q rank-bm25 datasets

import json, re, os
import numpy as np
from pathlib import Path
from datasets import load_dataset
from rank_bm25 import BM25Okapi

# ---- settings ----
NUM_EXAMPLES = 10      # change to 5 or 20 later if you want
TOP_K = 5
CLAUSES_PATH = Path("data/gdpr_clauses.jsonl")
OUT_DIR = Path("results"); OUT_DIR.mkdir(exist_ok=True)
OUT_FILE = OUT_DIR / "bm25_results.jsonl"

# ---- simple tokenizer ----
def simple_tokenize(text):
    text = (text or "").lower()
    text = re.sub(r"[^\w\s]", " ", text)
    toks = [w for w in text.split() if len(w)>0]
    return toks

# ---- load GDPR clauses ----
assert CLAUSES_PATH.exists(), f"{CLAUSES_PATH} not found. Re-run Week1 cell if missing."
clauses = []
clause_ids = []
with open(CLAUSES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        clause_ids.append(rec.get("id"))
        clauses.append(rec.get("text",""))

tokenized_corpus = [simple_tokenize(c) for c in clauses]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index built on {len(clauses)} GDPR clauses.")

# ---- load OPP-115 ----
ds = load_dataset("alzoubi36/opp_115")
print("Loaded OPP-115 splits:", {k: len(v) for k,v in ds.items()})

# ---- run retrieval on first N examples ----
results = []
for i in range(min(NUM_EXAMPLES, len(ds["train"]))):
    ex = ds["train"][i]
    query_text = ex.get("text") or ex.get("segment") or ""
    q_toks = simple_tokenize(query_text)
    scores = bm25.get_scores(q_toks)
    top_idx = np.argsort(-scores)[:TOP_K]
    hits = []
    for rank, idx in enumerate(top_idx, start=1):
        hits.append({
            "rank": rank,
            "clause_id": clause_ids[idx],
            "score": float(scores[idx]),
            "snippet": clauses[idx][:400].strip()
        })
    out = {
        "example_index": i,
        "query_text": query_text,
        "query_label": ex.get("label"),
        "bm25_hits": hits
    }
    results.append(out)
    # print a compact summary for the user
    print(f"\nExample {i+1} (label={ex.get('label')})")
    print("Query:", query_text[:200].strip(), "...")
    for h in hits:
        print(f"  {h['rank']}. {h['clause_id']} (score={h['score']:.3f})  {h['snippet'][:120]}...")
    print("-"*60)

# ---- save results for later ----
with open(OUT_FILE, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"\nSaved BM25 results for {len(results)} examples to {OUT_FILE}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


BM25 index built on 77 GDPR clauses.
Loaded OPP-115 splits: {'train': 2185, 'validation': 550, 'test': 697}

Example 1 (label=[3])
Query: ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to you ...
  1. Article 12 (score=11.857)  Transparent information, communication and modalities for the exercise of the rights of the data subject Section 2 Infor...
  2. Article 85 (score=11.091)  Processing and freedom of expression and information...
  3. Article 99 (score=10.651)  Entry into force and application Report error Logo We are a consulting company specialised in the fields of data protect...
  4. Article 8 (score=8.330)  Conditions applicable to child&#8217;s consent in relation to information society services...
  5. Article 13 (score=8.075)  Information to be provided where personal data are collected from the data subject...
-----------------

In [3]:
# Week 2 - A: Compare BM25 vs Dense retrieval for the same examples (one cell)
import sys
!{sys.executable} -m pip install -q sentence-transformers numpy

from sentence_transformers import SentenceTransformer, util
import json, re, os, numpy as np
from pathlib import Path
from datasets import load_dataset

# Settings
NUM_EXAMPLES = 10
TOP_K = 5
CLAUSES_PATH = Path("data/gdpr_clauses.jsonl")
BM25_RESULTS = Path("results/bm25_results.jsonl")
OUT_DIR = Path("results"); OUT_DIR.mkdir(exist_ok=True)
OUT_FILE = OUT_DIR / "dense_results.jsonl"

# Load BM25 saved results (to keep same queries/order)
assert BM25_RESULTS.exists(), "Run BM25 cell first to create results/bm25_results.jsonl"
bm25_entries = [json.loads(l) for l in open(BM25_RESULTS, "r", encoding="utf-8")]

# Load GDPR clauses
clauses = []
clause_ids = []
with open(CLAUSES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        clause_ids.append(rec.get("id"))
        clauses.append(rec.get("text",""))

# Load embedding model (light & fast)
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")  # change device to "cuda" if you have GPU access

# Precompute clause embeddings (persist to memory)
print("Encoding GDPR clauses (this may take 10-60s)...")
emb_clauses = model.encode(clauses, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
# Normalize for cosine
emb_clauses = emb_clauses / np.linalg.norm(emb_clauses, axis=1, keepdims=True)

# Function to compute top-k dense matches for a query
def dense_topk(query, k=5):
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    sims = (q_emb @ emb_clauses.T)[0]
    top_idx = np.argsort(-sims)[:k]
    return [(clause_ids[i], float(sims[i]), clauses[i][:300]) for i in top_idx]

# Compare for the same queries in bm25_entries
dense_results = []
print("\nComparing BM25 (saved) vs Dense (this run):\n")
for i, entry in enumerate(bm25_entries[:NUM_EXAMPLES]):
    query = entry["query_text"]
    bm25_hits = entry["bm25_hits"]
    dense_hits = dense_topk(query, k=TOP_K)
    dense_results.append({
        "example_index": entry["example_index"],
        "query_text": query,
        "query_label": entry.get("query_label"),
        "bm25_hits": bm25_hits,
        "dense_hits": [{"rank": r+1, "clause_id": cid, "score": scr, "snippet": snip} for r,(cid,scr,snip) in enumerate(dense_hits)]
    })
    # Print a compact side-by-side comparison for the user
    print(f"Example {i+1} (label={entry.get('query_label')})")
    print("Query (truncated):", query[:200].strip(), "...")
    print("  BM25 top1 ->", bm25_hits[0]["clause_id"], f"(score={bm25_hits[0]['score']:.3f})")
    print("  Dense top1 ->", dense_hits[0][0], f"(score={dense_hits[0][1]:.4f})")
    # If different, show small snippets
    if bm25_hits[0]["clause_id"] != dense_hits[0][0]:
        print("  BM25 top1 snippet:", bm25_hits[0]["snippet"][:140].strip(), "...")
        print("  Dense top1 snippet:", dense_hits[0][2][:140].strip(), "...")
    print("-"*72)

# Save dense results
with open(OUT_FILE, "w", encoding="utf-8") as f:
    for rec in dense_results:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"\nSaved dense results to {OUT_FILE}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Encoding GDPR clauses (this may take 10-60s)...


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.95s/it]



Comparing BM25 (saved) vs Dense (this run):

Example 1 (label=[3])
Query (truncated): ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to you ...
  BM25 top1 -> Article 12 (score=11.857)
  Dense top1 -> Article 13 (score=0.3402)
  BM25 top1 snippet: Transparent information, communication and modalities for the exercise of the rights of the data subject Section 2 Information and access to ...
  Dense top1 snippet: Information to be provided where personal data are collected from the data subject ...
------------------------------------------------------------------------
Example 2 (label=[9])
Query (truncated): (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their ...
  BM25 top1 -> Article 14 (score=

In [4]:
# Single cell: simple blended BM25 + dense retriever and demo for first 5 examples
import sys, os
!{sys.executable} -m pip install -q rank-bm25 sentence-transformers numpy

import json, re, numpy as np
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

# --- settings ---
CLAUSES_PATH = Path("data/gdpr_clauses.jsonl")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
EMB_DIR = Path("indices"); EMB_DIR.mkdir(exist_ok=True)
EMB_FILE = EMB_DIR / "emb_clauses.npy"
MODEL_NAME = "all-MiniLM-L6-v2"    # small, fast
NUM_EXAMPLES = 5
TOP_K = 5
ALPHA = 0.5  # weight for BM25 (1-ALPHA for dense)

# --- simple tokenizer used for BM25 ---
def simple_tokenize(text):
    t = (text or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    toks = [w for w in t.split() if len(w)>0]
    return toks

# --- load GDPR clauses ---
assert CLAUSES_PATH.exists(), "Missing data/gdpr_clauses.jsonl (run Week1 cells)"
clauses = []; clause_ids = []
with open(CLAUSES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        clause_ids.append(rec.get("id"))
        clauses.append(rec.get("text",""))

# --- build BM25 (fast, small corpus) ---
tokenized_corpus = [simple_tokenize(c) for c in clauses]
bm25 = BM25Okapi(tokenized_corpus)

# --- load or compute clause embeddings (cached) ---
if EMB_FILE.exists():
    emb_clauses = np.load(EMB_FILE)
    print("Loaded cached clause embeddings:", EMB_FILE)
else:
    print("Encoding GDPR clauses with", MODEL_NAME, "(this may take 10-60s)...")
    model = SentenceTransformer(MODEL_NAME, device="cpu")  # change to "cuda" if you have GPU available
    emb_clauses = model.encode(clauses, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
    # normalize embeddings for cosine similarity
    emb_clauses = emb_clauses / np.linalg.norm(emb_clauses, axis=1, keepdims=True)
    np.save(EMB_FILE, emb_clauses)
    print("Saved clause embeddings to:", EMB_FILE)

# if model not yet created, create it now for queries
try:
    model
except NameError:
    model = SentenceTransformer(MODEL_NAME, device="cpu")

# --- helpful normalization function ---
def minmax_normalize(arr):
    mn = float(np.min(arr))
    mx = float(np.max(arr))
    if mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

# --- blended retrieve function ---
def blended_retrieve(query, alpha=ALPHA, topk=TOP_K):
    # BM25 scores
    q_tokens = simple_tokenize(query)
    bm_scores = bm25.get_scores(q_tokens)        # length = num_clauses

    # Dense similarity
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    dense_sims = (q_emb @ emb_clauses.T)[0]      # cosine sims in [-1,1]

    # normalize both to [0,1]
    bm_n = minmax_normalize(bm_scores)
    dense_n = minmax_normalize(dense_sims)

    # blended score
    blended = alpha * bm_n + (1.0 - alpha) * dense_n

    # get topk indices
    top_idx = np.argsort(-blended)[:topk]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        results.append({
            "rank": rank,
            "clause_id": clause_ids[idx],
            "bm25_score": float(bm_scores[idx]),
            "dense_score": float(dense_sims[idx]),
            "blended_score": float(blended[idx]),
            "snippet": clauses[idx][:400].strip()
        })
    return results

# --- run blended retrieval for first N examples from OPP-115 train ---
ds = load_dataset("alzoubi36/opp_115")
print("Running blended retrieval for first", NUM_EXAMPLES, "training examples (alpha =", ALPHA, ")...\n")

for i in range(min(NUM_EXAMPLES, len(ds["train"]))):
    ex = ds["train"][i]
    query_text = ex.get("text") or ex.get("segment") or ""
    print(f"Example {i+1} (label={ex.get('label')})")
    print("Query:", query_text[:200].strip(), "...\n")
    blended_hits = blended_retrieve(query_text, alpha=ALPHA, topk=TOP_K)
    for h in blended_hits:
        print(f"  {h['rank']}. {h['clause_id']}  blended={h['blended_score']:.4f}  bm25={h['bm25_score']:.3f}  dense={h['dense_score']:.4f}")
        print("     ", h['snippet'][:200].strip(), "...")
    print("-"*72)

# --- optionally save blended results ---
out_path = RESULTS_DIR / "blended_results.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for i in range(min(NUM_EXAMPLES, len(ds["train"]))):
        rec = {
            "example_index": i,
            "query_text": ds["train"][i].get("text"),
            "query_label": ds["train"][i].get("label"),
            "blended_hits": blended_retrieve(ds["train"][i].get("text"), alpha=ALPHA, topk=TOP_K)
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("\nSaved blended results to", out_path)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Encoding GDPR clauses with all-MiniLM-L6-v2 (this may take 10-60s)...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.46s/it]


Saved clause embeddings to: indices/emb_clauses.npy
Running blended retrieval for first 5 training examples (alpha = 0.5 )...

Example 1 (label=[3])
Query: ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to you ...

  1. Article 12  blended=0.9062  bm25=11.857  dense=0.2688
      Transparent information, communication and modalities for the exercise of the rights of the data subject Section 2 Information and access to personal data ...
  2. Article 13  blended=0.8405  bm25=8.075  dense=0.3402
      Information to be provided where personal data are collected from the data subject ...
  3. Article 8  blended=0.7688  bm25=8.330  dense=0.2774
      Conditions applicable to child&#8217;s consent in relation to information society services ...
  4. Article 14  blended=0.7480  bm25=7.449  dense=0.2899
      Information to be provided where personal 

In [5]:
# ONE CELL: add decision-aware boost to blended retrieval and demo (tiny)
import numpy as np, json
from pathlib import Path

# --- Config (tune lambda_ to change boost strength) ---
LAMBDA = 0.15   # normative boost weight (try 0.05, 0.15, 0.3)
TOP_K = 5
NUM_EXAMPLES = 5

# --- safety: assume variables from previous blended cell exist; if not, try to restore ---
# required: clauses, clause_ids, bm25, emb_clauses, model, simple_tokenize
if 'clauses' not in globals():
    # try to load clauses from file
    CLAUSES_PATH = Path("data/gdpr_clauses.jsonl")
    clauses=[]; clause_ids=[]
    with open(CLAUSES_PATH,"r",encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            clause_ids.append(rec.get("id"))
            clauses.append(rec.get("text",""))
if 'emb_clauses' not in globals():
    # try to load cached embeddings if available
    EMB_FILE = Path("indices/emb_clauses.npy")
    if EMB_FILE.exists():
        import numpy as _np
        emb_clauses = _np.load(str(EMB_FILE))
    else:
        raise RuntimeError("Clause embeddings not found. Run the blended cell to create cached embeddings first.")

# helper: min-max normalize
def minmax_normalize(arr):
    mn = float(np.min(arr)); mx = float(np.max(arr))
    if mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

# normative scoring function (simple, interpretable)
NORMATIVE_TERMS = [
    "shall","must","must not","shall not","required","required to","prohibit",
    "is prohibited","only if","unless","except","penalty","liable","consent",
    "obligation","obliged","comply","compliance"
]
def normative_score(text):
    t = (text or "").lower()
    # count distinct terms occurrences (cap at 5 to avoid runaway)
    count = 0
    for term in NORMATIVE_TERMS:
        if term in t:
            count += 1
    return min(count / 5.0, 1.0)  # scale to [0,1]

# decision-aware blended retriever (returns ranked list)
def blended_decision_retrieve(query, alpha=0.5, lambda_=LAMBDA, topk=TOP_K):
    # BM25 scores
    q_tokens = simple_tokenize(query)
    bm_scores = bm25.get_scores(q_tokens)            # shape: (num_clauses,)

    # Dense sims (cosine) using cached clause embeddings and model
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    dense_sims = (q_emb @ emb_clauses.T)[0]

    # normalize
    bm_n = minmax_normalize(bm_scores)
    dense_n = minmax_normalize(dense_sims)

    # blended base
    blended = alpha * bm_n + (1.0 - alpha) * dense_n

    # compute decision-aware score by adding lambda * normative_score(clause_text)
    decision_scores = blended.copy()
    # vectorize normative scoring for speed
    norms = np.array([normative_score(txt) for txt in clauses], dtype=float)
    decision_scores = blended + lambda_ * norms

    # select topk
    top_idx = np.argsort(-decision_scores)[:topk]
    out = []
    for rank, idx in enumerate(top_idx, start=1):
        out.append({
            "rank": rank,
            "clause_id": clause_ids[idx],
            "bm25_raw": float(bm_scores[idx]),
            "dense_raw": float(dense_sims[idx]),
            "bm25_norm": float(bm_n[idx]),
            "dense_norm": float(dense_n[idx]),
            "blended": float(blended[idx]),
            "normative": float(norms[idx]),
            "decision_score": float(decision_scores[idx]),
            "snippet": clauses[idx][:500].strip()
        })
    return out

# --- Demo: print blended (old) vs decision-aware (new) top-3 for first NUM_EXAMPLES queries ---
print("Decision-aware blended demo (lambda = {:.3f})\n".format(LAMBDA))

ds = load_dataset("alzoubi36/opp_115")
for i in range(min(NUM_EXAMPLES, len(ds["train"]))):
    ex = ds["train"][i]
    q = ex.get("text") or ex.get("segment") or ""
    print(f"Example {i+1} (label={ex.get('label')})")
    print("Query:", q[:200].strip(), "...\n")

    # old blended (reuse your blended_retrieve if present) - fallback compute here
    try:
        old = blended_retrieve(q, alpha=0.5, topk=3)
    except Exception:
        # compute blended quickly here (same logic without normative boost)
        bm_scores = bm25.get_scores(simple_tokenize(q))
        q_emb = model.encode([q], convert_to_numpy=True)
        q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
        dense_sims = (q_emb @ emb_clauses.T)[0]
        bm_n = minmax_normalize(bm_scores)
        dense_n = minmax_normalize(dense_sims)
        blended_base = 0.5 * bm_n + 0.5 * dense_n
        top_idx_old = np.argsort(-blended_base)[:3]
        old = [{"rank": r+1, "clause_id": clause_ids[idx], "blended": float(blended_base[idx]), "snippet": clauses[idx][:200]} for r,idx in enumerate(top_idx_old)]

    new = blended_decision_retrieve(q, alpha=0.5, lambda_=LAMBDA, topk=3)

    print("  OLD blended top-3:")
    for h in old:
        print(f"    {h['rank']}. {h['clause_id']}  (blended={h.get('blended',0):.4f})")
    print("  NEW decision-aware top-3:")
    for h in new:
        print(f"    {h['rank']}. {h['clause_id']}  (decision_score={h['decision_score']:.4f}, normative={h['normative']:.3f})")
    print("-"*72)

# Optionally save decision-aware results for the first NUM_EXAMPLES
outp = Path("results/decision_blended.jsonl")
with open(outp, "w", encoding="utf-8") as f:
    for i in range(min(NUM_EXAMPLES, len(ds["train"]))):
        q = ds["train"][i].get("text")
        f.write(json.dumps({
            "example_index": i,
            "query_text": q,
            "query_label": ds["train"][i].get("label"),
            "decision_hits": blended_decision_retrieve(q, alpha=0.5, lambda_=LAMBDA, topk=TOP_K)
        }, ensure_ascii=False) + "\n")
print("\nSaved decision-aware blended results to", outp)


Decision-aware blended demo (lambda = 0.150)

Example 1 (label=[3])
Query: ""Contact Us"" Link If you contact us through the ""Contact Us"" link on this site, we ask you for information such as your first name, e-mail address, and other information, so we can respond to you ...

  OLD blended top-3:
    1. Article 12  (blended=0.0000)
    2. Article 13  (blended=0.0000)
    3. Article 8  (blended=0.0000)
  NEW decision-aware top-3:
    1. Article 12  (decision_score=0.9062, normative=0.000)
    2. Article 13  (decision_score=0.8405, normative=0.000)
    3. Article 8  (decision_score=0.7988, normative=0.200)
------------------------------------------------------------------------
Example 2 (label=[9])
Query: (ii) You have entered a contest or sweepstakes sponsored by a third party, in which case the information you provide via the contest or sweepstakes may be shared by us with that third party for their ...

  OLD blended top-3:
    1. Article 14  (blended=0.0000)
    2. Article 12  (b

In [7]:
# Accuracy comparison for BM25 vs Dense vs Blended (FIXED)

import numpy as np
from datasets import load_dataset

# ---- Settings ----
N = 100  # number of examples to evaluate
ALPHA = 0.5

# Expected GDPR articles per OPP label (proxy ground truth)
EXPECTED = {
    3: {"Article 12", "Article 13"},        # collection
    9: {"Article 12", "Article 14"},        # sharing
    "3_9": {"Article 12", "Article 13", "Article 14"}
}

def expected_set(label):
    if label == [3]:
        return EXPECTED[3]
    if label == [9]:
        return EXPECTED[9]
    if label == [3,9] or label == [9,3]:
        return EXPECTED["3_9"]
    return set()  # skip other labels

def minmax(arr):
    mn, mx = np.min(arr), np.max(arr)
    if mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

# Load dataset properly
ds = load_dataset("alzoubi36/opp_115")
examples = ds["train"].select(range(N))

bm25_correct = 0
dense_correct = 0
blend_correct = 0
valid = 0

for ex in examples:
    label = ex["label"]
    exp = expected_set(label)
    if not exp:
        continue
    valid += 1
    query = ex["text"]

    # ---- BM25 ----
    bm_scores = bm25.get_scores(simple_tokenize(query))
    bm_top = clause_ids[np.argmax(bm_scores)]

    # ---- Dense ----
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    dense_scores = (q_emb @ emb_clauses.T)[0]
    dense_top = clause_ids[np.argmax(dense_scores)]

    # ---- Blended ----
    bm_n = minmax(bm_scores)
    dense_n = minmax(dense_scores)
    blend_scores = ALPHA * bm_n + (1 - ALPHA) * dense_n
    blend_top = clause_ids[np.argmax(blend_scores)]

    if bm_top in exp:
        bm25_correct += 1
    if dense_top in exp:
        dense_correct += 1
    if blend_top in exp:
        blend_correct += 1

print(f"Evaluated {valid} examples\n")
print(f"BM25 accuracy   : {bm25_correct / valid:.3f}")
print(f"Dense accuracy  : {dense_correct / valid:.3f}")
print(f"Blended accuracy: {blend_correct / valid:.3f}")


Evaluated 50 examples

BM25 accuracy   : 0.420
Dense accuracy  : 0.560
Blended accuracy: 0.680


In [9]:
# Retrieval evaluation: multiple metrics (one cell)
import json, numpy as np
from pathlib import Path
from datasets import load_dataset

# ---- Settings ----
N = 200                 # number of train examples to evaluate (change as you like)
TOP_KS = [1, 3, 5]
TARGET_ARTS = {"Article 12", "Article 13", "Article 14"}
# label->expected mapping (keeps old mapping as one view, but we won't rely on it alone)
EXPECTED = {
    3: {"Article 12", "Article 13"},
    9: {"Article 12", "Article 14"},
    "3_9": {"Article 12", "Article 13", "Article 14"}
}

def expected_set(label):
    if label == [3]:
        return EXPECTED[3]
    if label == [9]:
        return EXPECTED[9]
    if label == [3,9] or label == [9,3]:
        return EXPECTED["3_9"]
    return set()

# ---- load queries ----
ds = load_dataset("alzoubi36/opp_115")
num_examples = min(N, len(ds["train"]))
queries = [ds["train"][i] for i in range(num_examples)]

# ---- ensure BM25 and dense resources exist; otherwise error with guidance ----
missing = []
for v in ("bm25","emb_clauses","model","clause_ids","clauses"):
    if v not in globals():
        missing.append(v)
if missing:
    raise RuntimeError(f"Missing variables in notebook: {missing}. Run the BM25/blended/dense cells first to define them.")

# ---- helper functions ----
def minmax(arr):
    mn, mx = float(np.min(arr)), float(np.max(arr))
    if mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def topk_from_bm25(query, k):
    scores = bm25.get_scores(simple_tokenize(query))
    idx = np.argsort(-scores)[:k]
    return [clause_ids[i] for i in idx], [float(scores[i]) for i in idx]

def topk_from_dense(query, k):
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    sims = (q_emb @ emb_clauses.T)[0]
    idx = np.argsort(-sims)[:k]
    return [clause_ids[i] for i in idx], [float(sims[i]) for i in idx]

def topk_from_blend(query, k, alpha=0.5):
    # bm25 raw and dense raw
    bm_scores = bm25.get_scores(simple_tokenize(query))
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    dense_sims = (q_emb @ emb_clauses.T)[0]
    bm_n = minmax(bm_scores)
    dense_n = minmax(dense_sims)
    blend = alpha * bm_n + (1 - alpha) * dense_n
    idx = np.argsort(-blend)[:k]
    return [clause_ids[i] for i in idx], [float(blend[i]) for i in idx]

# ---- accumulators ----
counts = {
    "bm25_top1_label_correct": 0,
    "dense_top1_label_correct": 0,
    "blend_top1_label_correct": 0,
    "examples_with_label_mapping": 0
}
# recall@k for target arts
recall = {"bm25": {k:0 for k in TOP_KS}, "dense": {k:0 for k in TOP_KS}, "blend": {k:0 for k in TOP_KS}}
# MRR for target arts
mrr = {"bm25": 0.0, "dense": 0.0, "blend": 0.0}
# agreement counts
agree_top1 = {"bm25_dense":0, "bm25_blend":0, "dense_blend":0}
agree_topk = {"bm25_dense":0, "bm25_blend":0, "dense_blend":0}
# avg normalized rank for target arts
norm_ranks = {"bm25": [], "dense": [], "blend": []}
# coverage: set of clause ids seen in any top-5
coverage = {"bm25": set(), "dense": set(), "blend": set()}
# per-label breakdown
per_label = {}

# helper for reciprocal rank of target arts
def reciprocal_rank(top_list, target_set):
    for i, cid in enumerate(top_list):
        if cid in target_set:
            return 1.0/(i+1)
    return 0.0

# evaluate loop
for ex in queries:
    qtext = ex.get("text") or ex.get("segment") or ""
    label = ex.get("label")
    exp_set = expected_set(label)
    if exp_set:
        counts["examples_with_label_mapping"] += 1

    # top-k lists
    bm_ids, bm_scores = topk_from_bm25(qtext, max(TOP_KS))
    de_ids, de_scores = topk_from_dense(qtext, max(TOP_KS))
    bl_ids, bl_scores = topk_from_blend(qtext, max(TOP_KS), alpha=0.5)

    # top1 label-based accuracy (only when mapping exists)
    if exp_set:
        if bm_ids[0] in exp_set:
            counts["bm25_top1_label_correct"] += 1
        if de_ids[0] in exp_set:
            counts["dense_top1_label_correct"] += 1
        if bl_ids[0] in exp_set:
            counts["blend_top1_label_correct"] += 1

    # recall@k for target arts
    for k in TOP_KS:
        if any(cid in TARGET_ARTS for cid in bm_ids[:k]):
            recall["bm25"][k] += 1
        if any(cid in TARGET_ARTS for cid in de_ids[:k]):
            recall["dense"][k] += 1
        if any(cid in TARGET_ARTS for cid in bl_ids[:k]):
            recall["blend"][k] += 1

    # MRR for target arts
    mrr["bm25"] += reciprocal_rank(bm_ids, TARGET_ARTS)
    mrr["dense"] += reciprocal_rank(de_ids, TARGET_ARTS)
    mrr["blend"] += reciprocal_rank(bl_ids, TARGET_ARTS)

    # agreement top1
    if bm_ids[0] == de_ids[0]:
        agree_top1["bm25_dense"] += 1
    if bm_ids[0] == bl_ids[0]:
        agree_top1["bm25_blend"] += 1
    if de_ids[0] == bl_ids[0]:
        agree_top1["dense_blend"] += 1
    # agreement@5 (set overlap)
    bm_set = set(bm_ids[:5]); de_set = set(de_ids[:5]); bl_set = set(bl_ids[:5])
    if len(bm_set & de_set) > 0:
        agree_topk["bm25_dense"] += 1
    if len(bm_set & bl_set) > 0:
        agree_topk["bm25_blend"] += 1
    if len(de_set & bl_set) > 0:
        agree_topk["dense_blend"] += 1

    # normalized rank for target arts (if present in top-5), else 6 (penalty)
    def norm_rank(top_list):
        found = None
        for i,cid in enumerate(top_list[:5]):
            if cid in TARGET_ARTS:
                found = i+1
                break
        if found is None:
            return None
        # normalize to [0,1] where 1 means top1, 0 means rank 5
        return 1.0 - ((found-1)/4.0)

    nr = norm_rank(bm_ids)
    if nr is not None: norm_ranks["bm25"].append(nr)
    nr = norm_rank(de_ids)
    if nr is not None: norm_ranks["dense"].append(nr)
    nr = norm_rank(bl_ids)
    if nr is not None: norm_ranks["blend"].append(nr)

    # coverage accumulation
    coverage["bm25"].update(bm_ids[:5])
    coverage["dense"].update(de_ids[:5])
    coverage["blend"].update(bl_ids[:5])

    # per-label stats
    lab_key = str(label)
    if lab_key not in per_label:
        per_label[lab_key] = {"count":0, "bm25_correct":0, "dense_correct":0, "blend_correct":0}
    per_label[lab_key]["count"] += 1
    if exp_set:
        if bm_ids[0] in exp_set:
            per_label[lab_key]["bm25_correct"] += 1
        if de_ids[0] in exp_set:
            per_label[lab_key]["dense_correct"] += 1
        if bl_ids[0] in exp_set:
            per_label[lab_key]["blend_correct"] += 1

# finalize metrics
num = len(queries)
eval_valid = counts["examples_with_label_mapping"]
metrics = {
    "num_evaluated": num,
    "num_label_mapped_examples": eval_valid,
    "top1_label_accuracy": {
        "bm25": counts["bm25_top1_label_correct"] / eval_valid if eval_valid else None,
        "dense": counts["dense_top1_label_correct"] / eval_valid if eval_valid else None,
        "blend": counts["blend_top1_label_correct"] / eval_valid if eval_valid else None
    },
    "recall_at_k_target": {
        "bm25": {k: recall["bm25"][k]/num for k in TOP_KS},
        "dense": {k: recall["dense"][k]/num for k in TOP_KS},
        "blend": {k: recall["blend"][k]/num for k in TOP_KS},
    },
    "mrr_target": {k: mrr[k]/num for k in mrr},
    "agreement_top1": {k: agree_top1[k]/num for k in agree_top1},
    "agreement_top5": {k: agree_topk[k]/num for k in agree_topk},
    "avg_norm_rank_target": {k: (np.mean(norm_ranks[k]) if len(norm_ranks[k])>0 else None) for k in norm_ranks},
    "coverage_top5_unique": {k: len(coverage[k])/len(clause_ids) for k in coverage},
    "per_label": per_label
}

# print a compact summary
print("Evaluated examples:", num)
print("Label-mapped examples (for top1 label-accuracy):", eval_valid)
print("Top-1 label accuracy (proxy):", metrics["top1_label_accuracy"])
print("Recall@k (target arts):")
for k in TOP_KS:
    print(f"  k={k}: BM25={metrics['recall_at_k_target']['bm25'][k]:.3f}, Dense={metrics['recall_at_k_target']['dense'][k]:.3f}, Blend={metrics['recall_at_k_target']['blend'][k]:.3f}")
print("MRR (target arts):", {k: f"{metrics['mrr_target'][k]:.3f}" for k in metrics['mrr_target']})
print("Agreement top1:", {k: f"{metrics['agreement_top1'][k]:.3f}" for k in metrics['agreement_top1']})
print("Agreement top5 (overlap):", {k: f"{metrics['agreement_top5'][k]:.3f}" for k in metrics['agreement_top5']})
print("Average normalized rank (target in top-5):", metrics["avg_norm_rank_target"])
print("Coverage (how many unique clauses appear in top5):", metrics["coverage_top5_unique"])

# save results
Path("results").mkdir(exist_ok=True)
with open("results/retrieval_eval.json","w",encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print("\nSaved full metrics to results/retrieval_eval.json")


Evaluated examples: 200
Label-mapped examples (for top1 label-accuracy): 85
Top-1 label accuracy (proxy): {'bm25': 0.35294117647058826, 'dense': 0.5294117647058824, 'blend': 0.5764705882352941}
Recall@k (target arts):
  k=1: BM25=0.420, Dense=0.520, Blend=0.585
  k=3: BM25=0.695, Dense=0.850, Blend=0.920
  k=5: BM25=0.805, Dense=0.905, Blend=0.965
MRR (target arts): {'bm25': '0.566', 'dense': '0.690', 'blend': '0.750'}
Agreement top1: {'bm25_dense': '0.080', 'bm25_blend': '0.465', 'dense_blend': '0.315'}
Agreement top5 (overlap): {'bm25_dense': '0.810', 'bm25_blend': '0.995', 'dense_blend': '0.990'}
Average normalized rank (target in top-5): {'bm25': np.float64(0.765527950310559), 'dense': np.float64(0.8425414364640884), 'blend': np.float64(0.8536269430051814)}
Coverage (how many unique clauses appear in top5): {'bm25': 0.6883116883116883, 'dense': 0.7532467532467533, 'blend': 0.6883116883116883}

Saved full metrics to results/retrieval_eval.json


In [5]:
# 1) set HF token (interactive, secure)
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("hpaste ").strip()
print("HF_TOKEN set for this notebook session.")



HF_TOKEN set for this notebook session.


In [6]:
# 2) install required libs
import sys, subprocess
pkgs = [
    "transformers>=4.31.0",
    "accelerate>=0.20.3",
    "bitsandbytes>=0.41.1",
    "safetensors",
    "huggingface_hub"
]
print("Installing:", pkgs)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)
print("Install done. NOW: restart the Jupyter kernel (Kernel -> Restart).")


Installing: ['transformers>=4.31.0', 'accelerate>=0.20.3', 'bitsandbytes>=0.41.1', 'safetensors', 'huggingface_hub']
  Using cached huggingface_hub-1.2.3-py3-none-any.whl.metadata (13 kB)
Install done. NOW: restart the Jupyter kernel (Kernel -> Restart).


In [7]:
# 3) check HF token & model repo access
import os
from huggingface_hub import whoami, model_info

HF = os.environ.get("HF_TOKEN")
print("HF_TOKEN present:", bool(HF))

if HF:
    try:
        acct = whoami(api_token=HF)
        print("HF account:", acct.get("name") or acct.get("user"))
    except Exception as e:
        print("whoami failed:", type(e).__name__, str(e)[:300])

MODEL = "meta-llama/Llama-2-7b-chat-hf"
try:
    info = model_info(MODEL, token=HF)
    print("Model accessible. Sample files:")
    for f in info.siblings[:10]:
        print(" -", f.rfilename)
except Exception as e:
    print("model_info failed:", type(e).__name__, str(e)[:400])
    raise


HF_TOKEN present: True
whoami failed: TypeError HfApi.whoami() got an unexpected keyword argument 'api_token'
Model accessible. Sample files:
 - .gitattributes
 - LICENSE.txt
 - README.md
 - USE_POLICY.md
 - config.json
 - generation_config.json
 - model-00001-of-00002.safetensors
 - model-00002-of-00002.safetensors
 - model.safetensors.index.json
 - pytorch_model-00001-of-00002.bin


In [8]:
# 4) robust model load: try 8-bit -> fp16 -> cpu fallback
import os, traceback, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "meta-llama/Llama-2-7b-chat-hf"
HF = os.environ.get("HF_TOKEN")
if not HF:
    raise RuntimeError("HF_TOKEN not set. Run Cell 1 again.")

print("torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())

# load tokenizer first
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_auth_token=HF)

strategies = [
    ("8bit device_map=auto", {"device_map":"auto","load_in_8bit":True,"trust_remote_code":True,"use_auth_token":HF}),
    ("fp16 device_map=auto", {"device_map":"auto","torch_dtype":torch.float16,"trust_remote_code":True,"use_auth_token":HF}),
    ("cpu fallback", {"device_map":"cpu","trust_remote_code":True,"use_auth_token":HF})
]

model = None
last_exc = None
for desc, kwargs in strategies:
    print(f"\nTrying strategy: {desc}")
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, **kwargs)
        print("Loaded model with strategy:", desc)
        break
    except Exception as e:
        print("Strategy failed:", desc, " — ", type(e).__name__, str(e)[:300])
        traceback.print_exc()
        last_exc = e

if model is None:
    print("All strategies failed. Diagnostics (partial):")
    diag = {"torch_version": torch.__version__, "cuda_available": torch.cuda.is_available(), "hf_token_set": bool(HF)}
    print(json.dumps(diag, indent=2))
    raise last_exc

# move to eval
model.eval()
print("Model loaded and ready. Model device map:", getattr(model, "hf_device_map", None))


torch: 2.5.1 CUDA available: True
Loading tokenizer...


/home/saranyas/.conda/envs/torch-env/lib/python3.10/site-packages/transformers/models/auto/tokenization_auto.py:1041: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(



Trying strategy: 8bit device_map=auto


/home/saranyas/.conda/envs/torch-env/lib/python3.10/site-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:24<00:00, 12.27s/it]


Loaded model with strategy: 8bit device_map=auto
Model loaded and ready. Model device map: {'': 0}


In [9]:
# 5) single generation test (deterministic) — small prompt
import json, re, torch

# small system + example instruction
SYSTEM = "You are a GDPR compliance assistant. Return exactly one JSON: {\"verdict\":\"PASS|FAIL|RISK\",\"confidence\":0-1,\"explanation\":\"<=30 words\"}."

# example policy and clauses (use your retrieval file)
from pathlib import Path
rec_file = Path("results/blended_results.jsonl")
if not rec_file.exists():
    rec_file = Path("results/bm25_results.jsonl")
rec = json.loads(rec_file.read_text(encoding="utf-8").splitlines()[0])

policy_text = rec.get("query_text") or rec.get("query") or ""
hits = rec.get("blended_hits") or rec.get("bm25_hits") or []
clauses_text = "\n".join([
    f"{i+1}) {h['clause_id']}: {h['snippet'][:300].replace(chr(10), ' ')}"
    for i, h in enumerate(hits[:5])
])


prompt = SYSTEM + "\n\npolicy_text: \"" + policy_text.replace('"','\\"') + "\"\nretrieved_clauses:\n" + clauses_text + "\n\nAnswer:\n"

# tokenize & generate
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
with torch.no_grad():
    out_ids = model.generate(**inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
generated = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
print("Raw output:\n", generated[:1000], "\n")

# try parse JSON
parsed = None
try:
    parsed = json.loads(generated)
except Exception:
    m = re.search(r"\{.*\}", generated, re.S)
    parsed = json.loads(m.group(0)) if m else {"error":"no_json","raw":generated}

print("Parsed result:", parsed)



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw output:
 {
"verdict": "PASS",
"confidence": 0.8,
"explanation": "The policy text provides information on how to contact the organization, including the types of information that will be collected and how it will be used. The policy also informs users of their rights under GDPR, including the right to access their personal data and the right to withdraw their consent. The policy meets the requirements of Article 12, 13, 8, 14, and 67 of the GDPR."
} 

Parsed result: {'verdict': 'PASS', 'confidence': 0.8, 'explanation': 'The policy text provides information on how to contact the organization, including the types of information that will be collected and how it will be used. The policy also informs users of their rights under GDPR, including the right to access their personal data and the right to withdraw their consent. The policy meets the requirements of Article 12, 13, 8, 14, and 67 of the GDPR.'}


In [18]:
import pandas as pd
import json, re, torch, time

CSV_IN = "opp115_gdpr_sample_actual_filled.csv"
CSV_OUT = "opp115_gdpr_with_model_preds.csv"

df = pd.read_csv(CSV_IN)

def build_prompt(policy_text, retrieved_clauses):
    return f"""
You are a GDPR compliance assistant.
Be conservative. If unsure, choose RISK.

Return exactly one JSON object with keys:
verdict (PASS|FAIL|RISK), confidence (0-1).

Policy:
{policy_text}

Relevant GDPR clauses:
{retrieved_clauses}

Answer:
"""

def run_model(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=150, temperature=0.0, do_sample=False)
    text = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    # extract JSON
    try:
        match = re.search(r"\{.*\}", text, re.S)
        j = json.loads(match.group())
        return j.get("verdict", "RISK"), float(j.get("confidence", 0.5))
    except Exception:
        # conservative fallback
        return "RISK", 0.5

model_verdicts = []
model_confidences = []

for _, row in df.iterrows():
    prompt = build_prompt(row["policy_text"], row["retrieved_clauses"])
    verdict, conf = run_model(prompt)
    model_verdicts.append(verdict)
    model_confidences.append(conf)
    time.sleep(0.2)

df["model_verdict"] = model_verdicts
df["model_confidence"] = model_confidences

df.to_csv(CSV_OUT, index=False)
print("Saved predictions to:", CSV_OUT)


Saved predictions to: opp115_gdpr_with_model_preds.csv


In [32]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

CSV_PATH = "opp115_gdpr_with_model_preds.csv"
THRESHOLD = 0.92

df = pd.read_csv(CSV_PATH)

# normalize labels
df["model_verdict"] = df["model_verdict"].astype(str).str.upper().str.strip()
df["actual_verdict"] = df["actual_verdict"].astype(str).str.upper().str.strip()

# keep valid rows
valid = df[df["actual_verdict"].isin(["PASS", "FAIL", "RISK"])].copy()

# BEFORE
acc_before = (valid["model_verdict"] == valid["actual_verdict"]).mean()
print(f"Accuracy BEFORE downgrade: {acc_before:.3f}")

labels = ["PASS", "FAIL", "RISK"]
cm_before = confusion_matrix(valid["actual_verdict"], valid["model_verdict"], labels=labels)
print("\nConfusion Matrix BEFORE:")
display(pd.DataFrame(cm_before,
                     index=[f"Human_{l}" for l in labels],
                     columns=[f"Model_{l}" for l in labels]))

# APPLY DOWNGRADE
valid["model_verdict_adjusted"] = valid["model_verdict"]
mask = (valid["model_verdict"] == "PASS") & (valid["model_confidence"] < THRESHOLD)
valid.loc[mask, "model_verdict_adjusted"] = "RISK"

# AFTER
acc_after = (valid["model_verdict_adjusted"] == valid["actual_verdict"]).mean()
print(f"\nAccuracy AFTER downgrade (τ={THRESHOLD}): {acc_after:.3f}")

cm_after = confusion_matrix(valid["actual_verdict"], valid["model_verdict_adjusted"], labels=labels)
print("\nConfusion Matrix AFTER:")
display(pd.DataFrame(cm_after,
                     index=[f"Human_{l}" for l in labels],
                     columns=[f"Model_{l}" for l in labels]))

print("\nClassification Report AFTER:")
print(classification_report(
    valid["actual_verdict"],
    valid["model_verdict_adjusted"],
    labels=labels
))


Accuracy BEFORE downgrade: 0.200

Confusion Matrix BEFORE:


,Model_PASS,Model_FAIL,Model_RISK
Human_PASS,0,0,0
Human_FAIL,11,0,0
Human_RISK,13,0,6



Accuracy AFTER downgrade (τ=0.92): 0.600

Confusion Matrix AFTER:


,Model_PASS,Model_FAIL,Model_RISK
Human_PASS,0,0,0
Human_FAIL,0,0,11
Human_RISK,1,0,18



Classification Report AFTER:
              precision    recall  f1-score   support

        PASS       0.00      0.00      0.00         0
        FAIL       0.00      0.00      0.00        11
        RISK       0.62      0.95      0.75        19

    accuracy                           0.60        30
   macro avg       0.21      0.32      0.25        30
weighted avg       0.39      0.60      0.47        30



/home/saranyas/.conda/envs/torch-env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/saranyas/.conda/envs/torch-env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/saranyas/.conda/envs/torch-env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [38]:
import pandas as pd

CSV_PATH = "opp115_gdpr_with_model_preds.csv"
THRESHOLD = 0.91

df = pd.read_csv(CSV_PATH)

# normalize labels
df["actual_verdict"] = df["actual_verdict"].astype(str).str.upper().str.strip()
df["model_verdict"] = df["model_verdict"].astype(str).str.upper().str.strip()

# apply downgrade rule
df["model_verdict_adjusted"] = df["model_verdict"]
mask = (df["model_verdict"] == "PASS") & (df["model_confidence"] < THRESHOLD)
df.loc[mask, "model_verdict_adjusted"] = "RISK"

# keep valid rows
valid = df[df["actual_verdict"].isin(["PASS", "FAIL", "RISK"])].copy()

# define cost function
def error_cost(actual, predicted):
    if actual == predicted:
        return 0
    # false PASS: predicted PASS but actually RISK or FAIL
    if predicted == "PASS" and actual in ["RISK", "FAIL"]:
        return 2
    # false RISK or false FAIL
    return 1

# compute costs
valid["error_cost"] = valid.apply(
    lambda r: error_cost(r["actual_verdict"], r["model_verdict_adjusted"]),
    axis=1
)

total_cost = valid["error_cost"].sum()
max_cost = 2 * len(valid)  # worst case: all false PASS

cost_sensitive_accuracy = 1 - (total_cost / max_cost)

print(f"Total examples: {len(valid)}")

print(f"Cost-sensitive accuracy: {cost_sensitive_accuracy:.3f}")



Total examples: 30
Cost-sensitive accuracy: 0.783


In [ ]:

Classification Report:
              precision    recall  f1-score   support

        PASS       0.00      0.00      0.00         0
        FAIL       0.00      0.00      0.00        11
        RISK       1.00      0.32      0.48        19

    accuracy                           0.20        30
   macro avg       0.33      0.11      0.16        30
weighted avg       0.63      0.20      0.30        30

In [24]:
import pandas as pd

CSV_PATH = "opp115_gdpr_with_model_preds.csv"  # this is the cleaned file we saved

df = pd.read_csv(CSV_PATH)

display(df.head(30))


,example_id,policy_text,retrieved_clauses,model_verdict,model_confidence,actual_verdict,actual__confidence,correction_reason
0,481_fredericknewspost.com_34,"The Frederick News-Post and its owner, Randall...",Art.5 (Principles),PASS,0.8,RISK,0.5,NaN
1,1694_lids.com_112,Privacy Policy,Art.5 (Principles),PASS,0.8,FAIL,0.4,insufficient detail
2,686_military.com_168,Privacy Policy,Art.5 (Principles),PASS,0.8,FAIL,0.4,insufficient detail
3,1636_sidearmsports.com_257,SIDEARM Sports Privacy Policy,Art.5 (Principles),PASS,0.8,FAIL,0.4,insufficient detail
4,1636_sidearmsports.com_260,SIDEARM Sports Privacy Policy,Art.5 (Principles),PASS,0.8,FAIL,0.4,insufficient detail
5,1468_rockstargames.com_40,"Last updated on October 1, 2013",Art.5 (Principles),PASS,0.9,RISK,0.5,NaN
6,135_instagram.com_195,Privacy Policy,Art.5 (Principles),PASS,0.8,FAIL,0.4,insufficient detail
7,21_imdb.com_57,IMDb Privacy Notice,Art.5 (Principles),PASS,0.8,FAIL,0.4,insufficient detail
8,919_uh.edu_21,University of Houston Privacy Policy The Divis...,Art.5 (Principles),PASS,0.9,RISK,0.5,NaN
9,1099_enthusiastnetwork.com_44,EN: The Enthusiast Network (TEN or We or we) i...,Art.5 (Principles),RISK,0.5,RISK,0.5,NaN
